In [ ]:
# %pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="")
project = rf.workspace("test-3j2z9").project("test-ncscd")
version = project.version(1)
dataset = version.download("yolo26")



loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to test-1 in yolo26:: 100%|██████████| 404/404 [00:00<00:00, 9220.61it/s]


In [13]:
dataset.location

import yaml

# Path to your data.yaml
yaml_path = f"{dataset.location}/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Pulling the class names
classes = data.get('names')

print(f"Total classes: {len(classes)}")
print("Labels:", classes)

Total classes: 3
Labels: ['2door', 'door', 'window']


In [14]:
from ultralytics import YOLO

# 1. Load the YOLO26 Nano model (pretrained on COCO)
# Using the .pt file ensures we are fine-tuning, not training from scratch
model = YOLO('yolo26n.pt')

# 2. Fine-tune the model
# dataset.location was defined when you ran version.download("yolo26")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,      # Adjust based on your dataset size
    imgsz=640,       # Standard YOLO resolution
    plots=True,      # Generates training charts automatically
    device=0         # Use 0 for GPU, or 'cpu' if no GPU is available
)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/test-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

In [15]:
# Validate the model's performance on the validation set
metrics = model.val()

# Export for deployment (e.g., to ONNX for web/mobile use)
model.export(format='onnx')

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 910.1±268.8 MB/s, size: 29.3 KB)
val: Scanning /content/test-1/valid/labels.cache... 128 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 41.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.7it/s 3.0s
                   all        128       1252      0.771       0.38      0.418      0.241
                 2door         13         33          1          0     0.0368     0.0174
                  door        128        830      0.818      0.825      0.896      0.592
                window         29        389      0.494      0.314      0.321      0.115
Speed: 3.5ms preprocess, 5.9ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4

Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.


ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 7.8s, saved as '/content/runs/detect/train-2/weights/best.onnx' (9.4 MB)

Export complete (8.2s)
Results saved to /content/runs/detect/train-2/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/train-2/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/train-2/weights/best.onnx imgsz=640 data=/content/test-1/data.yaml  
Visualize:       https://netron.app


'/content/runs/detect/train-2/weights/best.onnx'